# Grievance Priority: EDA, labels, ML and explainability
**Run order:** run all cells top to bottom. Add your dataset in **cell 2**.

Data limits found in the screenshots, handled below:
- No date column: seasonal analysis is dropped. `complaint_id` is sequential, so it is used as the time proxy.
- No description text: Sub Category carries the detail instead of TF-IDF.
- The age column may be wrapped (`mm:ss` display). The age parsing cell checks it and warns.

In [ ]:
%pip install -q lightgbm shap scikit-learn scipy seaborn openpyxl

## 1. Dataset cell: add your file here

In [ ]:
# ======== ADD YOUR DATASET HERE ========
FILE = "grievances.xlsx"    # .xlsx / .xls / .csv path. In Colab leave as is and upload when asked.
# ========================================
import os
if not os.path.exists(FILE):
    try:
        from google.colab import files
        up = files.upload(); FILE = list(up.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{FILE} not found. Put it next to the notebook or change FILE.")

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, joblib, warnings
from scipy import stats
from scipy.stats import chi2_contingency, kruskal, boxcox
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

df = pd.read_excel(FILE) if FILE.lower().endswith(("xlsx","xls")) else pd.read_csv(FILE)
print(df.shape); df.head()

## 2. Cleaning (columns renamed by position, check the order)

In [ ]:
names = ["complaint_id","category","sub_category","grievance_age","ward","status","staff_remarks","staff_name"]
assert df.shape[1] >= 8, "Expected at least 8 columns"
df = df.iloc[:, :8].copy(); df.columns = names

df = df.replace("\\N", np.nan)                      # '\N' placeholder -> NaN
for c in ["category","sub_category","ward","status","staff_remarks","staff_name"]:
    df[c] = df[c].astype("string").str.strip()
df["category"] = df["category"].str.title()
df["staff_name"] = df["staff_name"].str.replace(r"(?i)marshal ward no", "Marshal Ward No", regex=True)
sp = df["staff_name"].str.rsplit("/", n=1, expand=True)
df["staff"] = sp[0].str.strip()
df["designation"] = sp[1].str.strip() if sp.shape[1] > 1 else pd.NA
df["complaint_id"] = pd.to_numeric(df["complaint_id"], errors="coerce")
df = df.dropna(subset=["complaint_id"]).drop_duplicates("complaint_id").sort_values("complaint_id").reset_index(drop=True)
print(df.shape); df.isnull().sum()

## 3. Grievance age parsing + sanity check

In [ ]:
def to_hours(v):
    if pd.isna(v): return np.nan
    if isinstance(v, pd.Timedelta): return v.total_seconds()/3600
    if hasattr(v, "hour"): return v.hour + v.minute/60 + v.second/3600
    if isinstance(v, (int, float, np.number)): return float(v)*24      # Excel day fraction
    p = str(v).split(":")
    try:
        if len(p) == 3: return int(p[0]) + float(p[1])/60 + float(p[2])/3600
        if len(p) == 2: return float(p[0])/60 + float(p[1])/3600        # 'mm:ss' -> hours
    except Exception: pass
    return np.nan

df["age_hours"] = df["grievance_age"].apply(to_hours)
print(df["age_hours"].describe())
if df["age_hours"].max() <= 1.0:
    print("\nWARNING: ages never exceed 1 hour -> values are wrapped 'mm:ss'. Re-export the column as General/[h]:mm:ss "
          "for real durations. Continuing, but treat age as a weak feature.")

## 4. Missing values

In [ ]:
print(df.isnull().sum())
df["remarks_missing"] = df["staff_remarks"].isna().astype(int)       # gap = still-open signal
print(pd.crosstab(df["status"], df["remarks_missing"]))
df["age_missing"] = df["age_hours"].isna().astype(int)
skew = df["age_hours"].skew()
df["age_hours"] = df["age_hours"].fillna(df["age_hours"].median() if abs(skew) > 0.5 else df["age_hours"].mean())
for c in ["category","sub_category","ward","status","designation"]:
    df[c] = df[c].fillna(df[c].mode()[0])
df["staff_remarks"] = df["staff_remarks"].fillna("None")

## 5. Univariate

In [ ]:
print(df["age_hours"].describe()); print("skew", round(df["age_hours"].skew(),2), "kurtosis", round(df["age_hours"].kurtosis(),2))
fig, ax = plt.subplots(1, 2, figsize=(12,3.5))
sns.histplot(df["age_hours"], kde=True, ax=ax[0]); sns.boxplot(x=df["age_hours"], ax=ax[1]); plt.show()
for c in ["category","status","staff_remarks","designation"]:
    print(f"\n{c}\n", df[c].value_counts().head(10))
    plt.figure(figsize=(8,3)); sns.countplot(y=df[c], order=df[c].value_counts().index[:10]); plt.title(c); plt.show()

## 6. Transformations (best per column by lowest |skew|)

In [ ]:
from sklearn.preprocessing import PowerTransformer
x = df["age_hours"].clip(lower=0)
cands = {"raw": x, "sqrt": np.sqrt(x), "log1p": np.log1p(x),
         "boxcox": pd.Series(boxcox(x + 1e-6)[0], index=x.index),
         "yeo-johnson": pd.Series(PowerTransformer("yeo-johnson").fit_transform(x.values.reshape(-1,1)).ravel(), index=x.index)}
sk = pd.Series({k: v.skew() for k, v in cands.items()}); best = sk.abs().idxmin()
print(sk.round(3), "\nBest:", best)
df["age_tf"] = cands[best]
fig, ax = plt.subplots(1, 2, figsize=(10,3.8))
stats.probplot(x, plot=ax[0]); ax[0].set_title("QQ before")
stats.probplot(df["age_tf"], plot=ax[1]); ax[1].set_title(f"QQ after ({best})"); plt.show()

## 7. Bivariate

In [ ]:
top = df["category"].value_counts().index[:8]; sub = df[df["category"].isin(top)]
plt.figure(figsize=(10,3.8)); sns.boxplot(data=sub, x="category", y="age_hours"); plt.xticks(rotation=30); plt.show()
groups = [g["age_hours"].values for _, g in sub.groupby("category") if len(g) > 1]
H, p = kruskal(*groups); print(f"Kruskal-Wallis age~category: H={H:.2f}, p={p:.3g}")
for a, b in [("category","status"), ("designation","status")]:
    ct = pd.crosstab(df[a], df[b]); chi2, p, dof, _ = chi2_contingency(ct)
    v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
    print(f"{a} vs {b}: chi2={chi2:.1f}, p={p:.3g}, Cramer's V={v:.2f}")
plt.figure(figsize=(8,4.5)); sns.heatmap(pd.crosstab(df["category"], df["status"], normalize="index"), annot=True, fmt=".2f", cmap="Blues"); plt.show()
num = df[["age_hours","remarks_missing"]].assign(closed=(df["status"]=="Closed").astype(int))
sns.heatmap(num.corr(), annot=True, cmap="coolwarm", center=0); plt.show()

## 8. Outliers: IQR + z-score\n**Decision:** `age_hours` is clipped at the IQR fences. Old complaints are real, so rows are kept, but extreme values would distort scaling.

In [ ]:
q1, q3 = df["age_hours"].quantile([.25,.75]); iqr = q3 - q1; lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
z = np.abs(stats.zscore(df["age_hours"])) if df["age_hours"].std() > 0 else np.zeros(len(df))
print(f"IQR [{lo:.2f},{hi:.2f}] outliers={((df.age_hours<lo)|(df.age_hours>hi)).sum()} | z>3: {(z>3).sum()}")
df["age_clipped"] = df["age_hours"].clip(lo, hi)

## 9. Ward hotspots (replaces the temporal section: no date column)

In [ ]:
ward = df.groupby("ward").agg(total=("complaint_id","count"), open_rate=("status", lambda s:(s!="Closed").mean()),
                              avg_age=("age_hours","mean")).sort_values("total", ascending=False)
print(ward.head(10))
ward.head(15)["total"].sort_values().plot.barh(figsize=(7,4.5), title="Top 15 wards by volume"); plt.show()
tw = ward.head(10).index
sns.heatmap(pd.crosstab(df[df.ward.isin(tw)]["ward"], df["category"]), cmap="YlOrRd", annot=True, fmt="d"); plt.show()
staff = df.groupby("staff").agg(cases=("complaint_id","count"), pending=("status", lambda s:(s!="Closed").sum())).sort_values("cases", ascending=False)
print(staff.head(10))

## 10. Priority label (documented rule)
`score = 0.5 * category_severity/4 + 0.3 * min(recurrence,10)/10 + 0.2 * ward_load_percentile`

- **Category severity** (1 to 4): Health 4; Solid Waste, Road, Veterinary 3; Electrical, Forest 2; Others 1; unknown 2.
- **Recurrence**: earlier complaints (lower ID) in the same ward and sub-category.
- **Ward load**: percentile of the ward's total complaint volume.
- **Season is excluded** because there is no date.
- Labels come from score percentiles: Low (bottom 50%), Medium (50-80%), High (80-95%), Critical (top 5%).

**Leakage rule:** the model never sees `category`, severity, recurrence or ward load, because they build the score.

In [ ]:
SEV = {"health":4, "solid waste":3, "road":3, "veterinary":3, "electrical":2, "forest":2, "others":1}
def sev(c):
    c = str(c).lower()
    return next((v for k, v in SEV.items() if k in c), 2)

df["severity"] = df["category"].apply(sev)
df["recurrence"] = df.groupby(["ward","sub_category"]).cumcount()          # sorted by id -> earlier complaints
df["ward_load"] = df["ward"].map(df["ward"].value_counts()).rank(pct=True)
df["score"] = 0.5*df["severity"]/4 + 0.3*df["recurrence"].clip(upper=10)/10 + 0.2*df["ward_load"]
LABELS = ["Low","Medium","High","Critical"]
df["priority"] = pd.cut(df["score"].rank(pct=True, method="first"), [0,.5,.8,.95,1.0], labels=LABELS, include_lowest=True)
print(df["priority"].value_counts().reindex(LABELS))
df["y"] = df["priority"].map({l:i for i,l in enumerate(LABELS)}).astype(int)

## 11. Features and splits

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

CAT = ["sub_category","ward","status","staff_remarks","designation"]   # raw attributes only
NUM = ["age_clipped","remarks_missing"]                                   # NOT: category, severity, recurrence, ward_load
X, y = df[CAT + NUM], df["y"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)   # stratified
cut = int(len(df)*0.8); Xtr_t, Xte_t, ytr_t, yte_t = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]  # time-proxy split
print("train/test:", Xtr.shape, Xte.shape, "| time split:", Xtr_t.shape, Xte_t.shape)

def make_pre():
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False), CAT),
        ("num", StandardScaler(), NUM)])

## 12. Baselines: Logistic Regression and Random Forest

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb

def evaluate(name, pipe, Xa, ya, Xb, yb):
    pipe.fit(Xa, ya); pr = pipe.predict(Xb)
    rep = classification_report(yb, pr, output_dict=True, zero_division=0)
    crit = rep.get("3", {}).get("recall", np.nan)
    return {"model": name, "macro_F1": f1_score(yb, pr, average="macro"), "Critical_recall": crit}

results = []
lr = Pipeline([("pre", make_pre()), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))])
rf = Pipeline([("pre", make_pre()), ("clf", RandomForestClassifier(300, class_weight="balanced", n_jobs=-1, random_state=42))])
results.append(evaluate("LogReg (baseline)", lr, Xtr, ytr, Xte, yte))
results.append(evaluate("RandomForest (baseline)", rf, Xtr, ytr, Xte, yte))
pd.DataFrame(results)

## 13. LightGBM, tuned with RandomizedSearchCV

In [ ]:
base = Pipeline([("pre", make_pre()), ("clf", lgb.LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1))])
param = {"clf__n_estimators":[150,300,500], "clf__learning_rate":[0.03,0.05,0.1], "clf__num_leaves":[15,31,63],
         "clf__min_child_samples":[10,20,40], "clf__subsample":[0.7,0.9,1.0], "clf__colsample_bytree":[0.6,0.8,1.0]}
skf = StratifiedKFold(5, shuffle=True, random_state=42)
search = RandomizedSearchCV(base, param, n_iter=12, scoring="f1_macro", cv=skf, random_state=42, n_jobs=-1)
search.fit(Xtr, ytr)
best = search.best_estimator_; print("Best params:", search.best_params_)
pr = best.predict(Xte)
results.append({"model":"LightGBM (tuned)", "macro_F1": f1_score(yte, pr, average="macro"),
                "Critical_recall": classification_report(yte, pr, output_dict=True, zero_division=0).get("3",{}).get("recall",np.nan)})
res = pd.DataFrame(results).round(3); print(res)
print("\n", classification_report(yte, pr, target_names=LABELS, zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(yte, pr), display_labels=LABELS).plot(cmap="Blues"); plt.show()
print("5-fold CV macro-F1 (tuned):", cross_val_score(best, Xtr, ytr, cv=skf, scoring="f1_macro").round(3))

## 14. Checks: time-based split and leakage test

In [ ]:
tb = Pipeline([("pre", make_pre()), ("clf", lgb.LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1,
              **{k.replace("clf__",""):v for k,v in search.best_params_.items()}))])
tb.fit(Xtr_t, ytr_t); print("Time-split macro-F1:", round(f1_score(yte_t, tb.predict(Xte_t), average="macro"),3))

# Leakage test: sub_category is a refinement of category, which builds the score. How much does the model rely on it?
CAT_NS = [c for c in CAT if c != "sub_category"]
pre_ns = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False), CAT_NS),
                            ("num", StandardScaler(), NUM)])
ns = Pipeline([("pre", pre_ns), ("clf", lgb.LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1))])
ns.fit(Xtr, ytr); print("Macro-F1 WITHOUT sub_category:", round(f1_score(yte, ns.predict(Xte), average="macro"),3),
      "(a big drop means the model mostly recovers category severity from sub_category)")

## 15. Explainable AI (SHAP) with plain-English reasons

In [ ]:
import shap
pre, clf = best.named_steps["pre"], best.named_steps["clf"]
Xt = pre.transform(Xte); fn = pre.get_feature_names_out()
expl = shap.TreeExplainer(clf); sv = expl.shap_values(Xt)
sv = np.stack(sv, axis=2) if isinstance(sv, list) else sv          # -> (n, features, classes)

glob = np.abs(sv).mean(axis=(0,2)); imp = pd.Series(glob, index=fn).sort_values(ascending=False).head(15)
imp.sort_values().plot.barh(figsize=(8,5), title="Global SHAP importance"); plt.show()

def readable(f):
    f = f.replace("cat__","").replace("num__","")
    for c in CAT:
        if f.startswith(c + "_"): return f"{c.replace('_',' ')} = {f[len(c)+1:]}"
    return {"age_clipped":"complaint age", "remarks_missing":"no staff remark yet"}.get(f, f)

def explain_row(i, k=3):
    p = int(best.predict(Xte.iloc[[i]])[0]); contrib = pd.Series(sv[i,:,p], index=fn)
    active = np.array([(not f.startswith("cat__")) or Xt[i, j] > 0.5 for j, f in enumerate(fn)])   # one-hot: only values this complaint HAS
    contrib = contrib[active].sort_values(ascending=False)
    why = [readable(f) for f in contrib.head(k).index if contrib[f] > 0]
    return f"{LABELS[p]}: " + (", ".join(why) if why else "no single dominant factor")

for i in range(5): print(i, "->", explain_row(i))

## 16. Save the model and a prediction function

In [ ]:
joblib.dump({"model": best, "labels": LABELS, "cat": CAT, "num": NUM}, "priority_model.joblib")

def predict_priority(rows: pd.DataFrame):
    """rows needs columns: sub_category, ward, status, staff_remarks, designation, age_clipped, remarks_missing"""
    m = joblib.load("priority_model.joblib")
    prob = m["model"].predict_proba(rows[m["cat"] + m["num"]])
    out = rows.copy(); out["priority"] = [m["labels"][i] for i in prob.argmax(1)]
    for j, l in enumerate(m["labels"]): out[f"p_{l}"] = prob[:, j].round(3)
    return out

predict_priority(Xte.head(5))

## 17. Insights (fill in after running)\n- Dominant categories and which stay open longest.\n- High-volume, high-open-rate wards to resource first.\n- Staff or designations carrying the heaviest backlog.\n- Limits: the label is rule-based, so the model learns the rule, not ground-truth urgency. No date column. Status and remarks overlap.